# 07 - A better TMD with no attack-aware training

**Rule for this notebook:** the defense learns nothing from attacks. No adversarial examples, no synonym lists, nothing from the attack code. Attacks are used only to *evaluate*, exactly as in notebooks 00-06. That is the same footing as the paper's TMD, so a win here is a like-for-like improvement of TMD.

**Idea.** TMD can only repair a sentence embedding that the attack has pushed off the manifold of clean sentences. A plainly fine-tuned BERT leans on a handful of words, so swapping a few of them moves the embedding far, and the projection cannot bring it back. Here BERT is first trained to give the same prediction, and nearly the same sentence embedding, when random words are masked, deleted or replaced by random words ("consistency training"). The noise has nothing to do with any attack: the positions are random and the replacements are random words from the training vocabulary, not synonyms. The paper's TMD then runs unchanged on top, with the paper's own GAN hyperparameters (Table 6).

| tag | language model | projection |
|---|---|---|
| `tmd_t6` | BERT from notebook 00 | TMD with the paper's Table 6 GAN (reference: TMD configured exactly as in the paper) |
| `nrb_mild` | noise-robust BERT, 10-30% of words noised per view | none |
| `nrb_strong` | noise-robust BERT, 30-60% of words noised per view | none |
| `<winner>_tmd` | the better of the two on validation | TMD with a Table 6 GAN trained on its embeddings |

**Selection protocol (fixed before any result):**
1. The two noise-robust BERTs are compared on the same 200 validation sentences as notebook 06 (TextFooler and BERT-Attack). The better mean wins, among those within 1 point of vanilla BERT's clean validation accuracy.
2. TMD is built on the winner. The final candidate is the winner alone or the winner + TMD, whichever has the better validation mean, among those within 1 point of our TMD's clean validation accuracy.
3. Only that candidate is attacked on the 1000 test sentences, once, with all three attacks, and compared with Table 1, with our TMD (McNemar test on the same sentences) and in the transfer test.

**Honesty notes:**
- The noisy views take their replacement words from the training set's vocabulary. Nothing else goes in.
- `<winner>_tmd` keeps TMD's randomness, which can confuse the attacks (our earlier finding on PWWS). The transfer test at the end shows how much of any gain holds without it.
- The noise-robust BERTs continue from notebook 00's model for 2 epochs at half its learning rate, rather than repeating the full 10-epoch fine-tuning.

**Cost (L4, rough):** about 1 h of fine-tuning, 1 h of validation attacks and 1.5-2 h for the test run, plus about 2 h for the optional reproduction run at the end. Everything is cached on Drive and resumes after a disconnect. Run notebooks 00 and 06 first.

In [ ]:
!pip -q install transformers datasets accelerate textattack tensorflow_hub
import os, sys, subprocess
REPO = 'https://github.com/shivanshgoyaliitk/for_shreyansh.git'
BRANCH = 'claude/magical-pasteur-ptp2k2'
DIR = '/content/for_shreyansh'
url = REPO
try:  # only needed if the repo is private: add a Colab secret named GITHUB_TOKEN
    from google.colab import userdata
    url = REPO.replace('https://', 'https://' + userdata.get('GITHUB_TOKEN') + '@')
except Exception:
    pass
if os.path.exists(DIR):
    subprocess.run(['git', '-C', DIR, 'pull', '-q'])
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, DIR], check=True)
sys.path.insert(0, DIR)

In [ ]:
import os, json, torch, pandas as pd, tmd_lab as T
T.setup_env()                      # the authors' TextAttack fork for every attack
from google.colab import drive
drive.mount('/content/drive')

DATASET = 'ag_news'
BASE_K = 30
WORK = '/content/drive/MyDrive/tmd_run_paperft'   # same folder as notebooks 00, 05 and 06
N_VAL = 200                        # validation sentences per attack (the same 200 as notebook 06)
N_TEST = 1000                      # the single final test run
SCREEN_ATTACKS = ('textfooler', 'bertattack')
ALL = ('textfooler', 'bertattack', 'pwws')
NOISE = {'nrb_mild': (0.1, 0.3), 'nrb_strong': (0.3, 0.6)}   # fraction of the words noised in each view
FT = dict(epochs=2, lr=1e-5, bs=32, w_kl=1.0, w_feat=1.0)    # consistency fine-tuning, starting from notebook 00's BERT
RUN_T6_TEST = True                 # last cell: TMD with the paper's Table 6 GAN on the test set (reproduction check)

lab = T.Lab(DATASET, work=WORK)
lab.finetune('base', **T.PAPER_FT)
E_tr, Y_tr = lab.embeddings('train', 'base')
E_va, Y_va = lab.embeddings('val', 'base')
E_te, Y_te = lab.embeddings('test', 'base')
gan = lab.get_gan('base', E_tr, E_va)          # our searched GAN, the one our TMD baseline uses
refs = {'vanilla': T.Victim(lab, mode='none'),
        'baseline': T.Victim(lab, gan=gan, mode='tmd', proj_mode='sample', k=BASE_K)}

## References on validation
Vanilla BERT and our TMD (cached from notebook 06), plus TMD with the paper's Table 6 GAN on the same BERT. Our own GAN search picked different hyperparameters from Table 6 (G learning rate 5e-4 vs 1e-4, 5 discriminator steps per generator step vs 1), so `tmd_t6` shows what TMD does when configured exactly as the paper reports.

In [ ]:
for tag, v in refs.items():
    T.evaluate(lab, v, tag, E_va, Y_va, attacks=SCREEN_ATTACKS, n=N_VAL, split='val', need_log=True)

gan_t6 = lab.get_gan('base', E_tr, E_va, use_search=False)   # Table 6 hyperparameters, trained once and cached
refs['tmd_t6'] = T.Victim(lab, gan=gan_t6, mode='tmd', proj_mode='sample', k=BASE_K)
T.evaluate(lab, refs['tmd_t6'], 'tmd_t6', E_va, Y_va, attacks=SCREEN_ATTACKS, n=N_VAL, split='val', need_log=True)
display(T.screen(lab, list(refs), N_VAL))

## Step 1: two noise-robust BERTs
Each training step sees a batch of clean training sentences and one noisy view of each. The loss is cross-entropy on both, plus a term that keeps the two predictions equal (KL divergence), plus a term that keeps the two sentence embeddings pointing the same way (cosine). The printed example shows what a noisy view looks like. After each epoch the log shows validation accuracy on clean sentences and on noisy views.

In [ ]:
texts_tr = lab.train_ds['text']
for tag, rate in NOISE.items():
    noise = T.NoiseAugmenter(texts_tr, rate=rate, seed=0)
    print(f'{tag} example view:\n  {texts_tr[0]}\n  {noise(texts_tr[0])}')
    lab.finetune_robust(tag, init='base', noise_fn=noise, **FT)

## Step 2: the noise-robust BERTs alone, on validation

In [ ]:
nrb = {}
for tag in NOISE:
    lab.use_model(tag)
    Et_va, _ = lab.embeddings('val', tag)
    nrb[tag] = T.Victim(lab, mode='none')
    T.evaluate(lab, nrb[tag], tag, Et_va, Y_va, attacks=SCREEN_ATTACKS, n=N_VAL, split='val', need_log=True)

bert_tab = T.screen(lab, ['vanilla'] + list(NOISE), N_VAL)
display(bert_tab)
van = bert_tab[bert_tab.variant == 'vanilla'].iloc[0]
ok = bert_tab[(bert_tab.variant != 'vanilla') & (bert_tab.clean >= van['clean'] - 1.0)]
assert len(ok), 'both noise-robust BERTs lose more than 1 point of clean accuracy: stop here and send the table'
best_bert = ok.loc[ok['mean'].idxmax(), 'variant']
gain = ok.loc[ok.variant == best_bert, 'mean'].item() - van['mean']
print(f'kept for TMD: {best_bert} ({gain:+.1f} points vs vanilla BERT, mean of the two attacks)')

## Step 3: the paper's TMD on top of the noise-robust BERT
New embeddings, and a GAN trained on them with the paper's Table 6 hyperparameters (no search). The projection is the paper's: the closest of 30 random samples.

In [ ]:
lab.use_model(best_bert)
Er_tr, _ = lab.embeddings('train', best_bert)
Er_va, _ = lab.embeddings('val', best_bert)
Er_te, _ = lab.embeddings('test', best_bert)
gan_r = lab.get_gan(best_bert, Er_tr, Er_va)
cands = {best_bert: nrb[best_bert],
         f'{best_bert}_tmd': T.Victim(lab, gan=gan_r, mode='tmd', proj_mode='sample', k=BASE_K)}
T.evaluate(lab, cands[f'{best_bert}_tmd'], f'{best_bert}_tmd', Er_va, Y_va, attacks=SCREEN_ATTACKS, n=N_VAL,
           split='val', need_log=True)
table = T.screen(lab, list(refs) + list(NOISE) + [f'{best_bert}_tmd'], N_VAL)
display(table)

In [ ]:
# Selection rule (fixed in advance): the better validation mean of TextFooler and BERT-Attack, among the two
# candidates within 1 point of our TMD's clean validation accuracy.
base = table[table.variant == 'baseline'].iloc[0]
cand = table[table.variant.isin(list(cands)) & (table.clean >= base['clean'] - 1.0)]
assert len(cand), 'no candidate keeps clean accuracy within 1 point of TMD: stop here and send the table'
chosen = cand.loc[cand['mean'].idxmax(), 'variant']
row = cand[cand.variant == chosen].iloc[0]
print('chosen variant:', chosen)
print(f"vs our TMD on validation: TextFooler {row['TextFooler'] - base['TextFooler']:+.1f}, "
      f"BERT-Attack {row['BERT-Attack'] - base['BERT-Attack']:+.1f}")
display(cand)
# notebook 07b starts its adversarial training from this BERT if it clearly beat vanilla BERT on validation
json.dump(dict(bert=best_bert, rate=NOISE[best_bert], gain_over_vanilla=gain), open(lab._path('nb07_choice.json'), 'w'))

## The one test run
The chosen variant on the 1000 test sentences (all three attacks), against the paper's Table 1, our TMD (McNemar test, same sentences) and the transfer test (texts that fooled vanilla BERT, fed once to each defense). Run this once.

In [ ]:
v = cands[chosen]
T.evaluate(lab, v, chosen, Er_te, Y_te, attacks=ALL, n=N_TEST, need_log=True)
display(T.compare(lab, chosen, baseline_tag='baseline', n=N_TEST))
display(pd.DataFrame([T.mcnemar(lab, 'baseline', chosen, a, N_TEST) for a in ALL]))
for attack in ALL:
    print('transfer from vanilla BERT,', attack)
    display(T.transfer_eval(lab, {'baseline': refs['baseline'], chosen: v}, 'vanilla', attack, N_TEST))

## Reproduction check: TMD with the paper's Table 6 GAN on the test set
Not part of the improvement claim. Our TMD's PWWS is 63.2 against the paper's 70.0, and our GAN differs from Table 6. This run shows whether the paper's own GAN settings close that gap. Set `RUN_T6_TEST = False` at the top to skip it (about 2 hours).

In [ ]:
if RUN_T6_TEST:
    T.evaluate(lab, refs['tmd_t6'], 'tmd_t6', E_te, Y_te, attacks=('pwws', 'textfooler', 'bertattack'), n=N_TEST,
               need_log=True)
    display(T.compare(lab, 'tmd_t6', baseline_tag='baseline', n=N_TEST))